# Detector Distance Comparison — FEP Scattering vs Crystal Signal
### SSA — Diamond I11 Beamline | Static Trial, 50/100/200mm

**Purpose:** A clean, focused notebook (separate from both prior notebooks) to test the original working theory: does moving the detector further back (100mm, 200mm) reduce FEP scattering, and if so, does that come at the cost of crystal (Bragg) signal? Both halves need answering together — one without the other would be a misleading result.

**Technique used:** Swish — now confirmed (Technique_Comparison_Report.docx) as the best-supported crystal-capture technique, so this comparison is done on the technique that will actually be used, not the 1000ms validation condition from Trial 1.

**Two questions, evaluated together:**
1. Does FEP ring intensity (from FEP_empty) actually fall with distance?
2. Does crystal spot_density (from Gly_AS_tri) hold up, or degrade, across the same distances?

---
**Run cells one at a time, top to bottom.**

---
## Step 0 — Paths, tracker, and all THREE geometries

In [ ]:
import os, glob
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════
# PATHS — confirm/update these before running
# ══════════════════════════════════════════════════════════════

MODULE_DIR   = r"C:/Users/ezxsa27/I11_PXRD_SSA/I11_PXRD_processing_scripts/FEP_Subtraction/"
import sys
sys.path.insert(0, MODULE_DIR)

RAW_ROOT     = r"E:/static/raw/raw_D1+2"          # <-- update if needed
OUTPUT_DIR   = r"E:/static/Corrected_2D/Distance_Comparison/"
TRACKER_PATH = r"C:/Users/ezxsa27/OneDrive - The University of Nottingham/Nottingham/Year 3 (June 26 -)/Static Trials/I11_Static_Trials_Tracker_CLEAN.xlsx"  # <-- update

# Three geometries - Day 1 calibrations, already validated (offset
# consistency + beam centre stability confirmed in Trial 1)
PONI_PATHS = {
    50:  r"E:/static/raw/Calib/Calib_50.poni",   # <-- update
    100: r"E:/static/raw/Calib/Calib_100.poni",  # <-- update
    200: r"E:/static/raw/Calib/Calib_200.poni",  # <-- update
}
MASK_PATHS = {
    50:  r"E:/static/raw/Calib/Calib_50_mask.npy",   # <-- update
    100: r"E:/static/raw/Calib/Calib_100_mask.npy",  # <-- update
    200: r"E:/static/raw/Calib/Calib_200_mask.npy",  # <-- update
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

tracker = pd.read_excel(TRACKER_PATH, sheet_name="Condition Tracker", header=6)
tracker.columns = [str(c).strip() for c in tracker.columns]
tracker["Sample code"]   = tracker["Sample code"].astype(str).str.strip()
tracker["Exposure code"] = tracker["Exposure code"].astype(str).str.strip()
tracker["Data logged?"]  = tracker["Data logged?"].astype(str).str.strip().str.upper()

tracker_clean = tracker[
    (tracker["Data logged?"] == "Y") &
    (tracker["Exclude?"].isna())
]

def collection_numbers_for(sample_code, distance, exposure_code=None):
    sub = tracker_clean[
        (tracker_clean["Sample code"] == sample_code) &
        (tracker_clean["Distance (mm)"] == distance)
    ]
    if exposure_code is not None:
        sub = sub[sub["Exposure code"] == exposure_code]
    return sub["Collection #(s)"].dropna().astype(int).astype(str).tolist()

def extract_number(fp):
    stem = os.path.splitext(os.path.basename(fp))[0]
    for part in reversed(stem.replace("-", "_").split("_")):
        if part.isdigit():
            return part
    return ""

all_raw_files = sorted(glob.glob(os.path.join(RAW_ROOT, "*.nxs")))
all_numbers = {f: extract_number(f) for f in all_raw_files}

def files_for_numbers(numbers):
    wanted = set(numbers)
    return sorted([f for f, n in all_numbers.items() if n in wanted])

print(f"Raw files found: {len(all_raw_files)}")
for dist in [50, 100, 200]:
    n_fep = len(collection_numbers_for("FEP_empty", dist, exposure_code="Swish"))
    n_gly = len(collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish"))
    print(f"{dist}mm: FEP_empty Swish = {n_fep} collection(s), Gly_AS_tri Swish = {n_gly} collection(s)")

In [ ]:
from fep_subtraction_2d_i11 import (
    beam_center_px_from_poni,
    FrameMetricConfig,
    evaluate_frame_with_status,
    _load_pixium_frame,
)
import h5py

def _read_exposure_time(filepath):
    """Static trial files use 'pixium_hdf' as the detector group name;
    also handles files where count_time stores multiple sub-exposure
    values by summing them."""
    paths = [
        "entry1/instrument/pixium_hdf/count_time",
        "entry/instrument/pixium_hdf/count_time",
        "entry1/instrument/detector/count_time",
        "entry1/instrument/detector/exposure_time",
        "entry/instrument/detector/count_time",
        "entry/instrument/detector/exposure_time",
        "entry1/count_time",
        "entry/count_time",
    ]
    with h5py.File(filepath, "r") as f:
        for p in paths:
            if p in f:
                val = np.squeeze(f[p][()])
                if val.ndim == 0:
                    return float(val)
                return float(np.sum(val))
    return None

# Load all three geometries
geometries = {}
for dist in [50, 100, 200]:
    cy, cx = beam_center_px_from_poni(PONI_PATHS[dist])
    mask = np.load(MASK_PATHS[dist])
    cfg = FrameMetricConfig(beam_center_px=(cy, cx), detect_fep_ring=True)
    geometries[dist] = {"cy": cy, "cx": cx, "mask": mask, "cfg": cfg}
    print(f"{dist}mm: beam centre row={cy:.2f} col={cx:.2f}, "
          f"mask {100*mask.sum()/mask.size:.2f}% masked ({mask.shape})")

print("\n✅ All three geometries loaded. Ready for Step 1.")

---
## Step 1 — FEP ring intensity vs distance (FEP_empty, Swish)

Tests the core theory: does FEP ring signal actually diminish as detector distance increases? Uses the same peak/noise approach as the original SNR sweep, but now across distance rather than exposure time. Ring radius is auto-detected per distance (position in pixels changes with distance; position in 2θ should not).

In [ ]:
from fep_subtraction_2d_i11 import _mad_sigma

def frame_signal_noise(filepaths, geom, ring_halfwidth=15, beam_excl=60, outer_excl=1200):
    """Average all provided frames, then measure peak FEP-ring signal
    and diffuse-region noise, auto-detecting the ring radius for this
    specific geometry."""
    stack = []
    total_time = 0.0
    for fp in filepaths:
        frame = _load_pixium_frame(fp)
        if frame.ndim == 3:
            frame = frame.mean(axis=0)   # average sub-frames within a multi-shot file
        stack.append(frame.astype(np.float64))
        t = _read_exposure_time(fp)
        total_time += t if t else 0.0

    arr = np.mean(stack, axis=0)
    mask = geom["mask"]
    arr[mask.astype(bool)] = 0.0

    rows, cols = arr.shape
    cy, cx = geom["cy"], geom["cx"]
    y_idx, x_idx = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x_idx - cx) ** 2 + (y_idx - cy) ** 2).astype(np.int32)
    r_max = int(r_map.max())

    radial_sum = np.bincount(r_map.ravel(), weights=arr.ravel(), minlength=r_max + 1)
    radial_cnt = np.bincount(r_map.ravel(), minlength=r_max + 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        radial_mean = np.where(radial_cnt > 0, radial_sum / radial_cnt, 0.0)

    # Auto-detect ring radius for THIS geometry (search a wide window,
    # since pixel position varies a lot across 50/100/200mm)
    search_lo, search_hi = 60, min(600, r_max)
    ring_r = int(np.argmax(radial_mean[search_lo:search_hi]) + search_lo)

    lo, hi = max(0, ring_r - ring_halfwidth), min(r_max, ring_r + ring_halfwidth)
    signal = float(np.mean(radial_mean[lo:hi + 1]))

    residual = arr - radial_mean[r_map]
    detection_mask = (r_map >= beam_excl) & (r_map <= outer_excl)
    detection_mask &= ~((r_map >= lo) & (r_map <= hi))
    detection_mask &= ~mask.astype(bool)
    noise = _mad_sigma(residual[detection_mask])

    snr = signal / noise if noise > 0 else float("nan")
    return {
        "ring_r_px": ring_r, "signal": signal, "noise": noise, "snr": snr,
        "total_time_s": total_time, "n_frames": len(stack),
    }

fep_results = []
for dist in [50, 100, 200]:
    numbers = collection_numbers_for("FEP_empty", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    if not files:
        print(f"⚠️  No FEP_empty Swish files found at {dist}mm — skipping")
        continue
    r = frame_signal_noise(files, geometries[dist])
    r["distance_mm"] = dist
    fep_results.append(r)
    print(f"{dist}mm: ring_r={r['ring_r_px']}px  signal={r['signal']:.1f}  "
          f"noise={r['noise']:.3f}  SNR={r['snr']:.2f}  (n={r['n_frames']} files, {r['total_time_s']:.1f}s total)")

df_fep = pd.DataFrame(fep_results)
print("\n✅ FEP comparison across distances complete.")

---
## Step 2 — Crystal spot_density vs distance (Gly_AS_tri, Swish)

The other half of the question: does crystal-capture quality hold up at longer distances, or degrade alongside (or despite) any FEP reduction? Evaluates every Swish frame at each distance individually — same per-frame approach as the technique comparison, not averaged.

In [ ]:
crystal_results = []

for dist in [50, 100, 200]:
    numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    if not files:
        print(f"⚠️  No Gly_AS_tri Swish files found at {dist}mm — skipping")
        continue

    cfg = geometries[dist]["cfg"]
    mask = geometries[dist]["mask"]

    for fp in files:
        try:
            probe = _load_pixium_frame(fp)
        except Exception as exc:
            print(f"⚠️  {os.path.basename(fp)}: failed to load ({exc}) — skipping")
            continue

        n_subframes = probe.shape[0] if probe.ndim == 3 else 1
        for sub_idx in range(n_subframes):
            metrics, status = evaluate_frame_with_status(
                fp, frame_index=sub_idx, mask=mask, config=cfg
            )
            if status.status != "ok":
                continue
            crystal_results.append({
                "distance_mm": dist,
                "file": os.path.basename(fp),
                "sub_frame": sub_idx,
                "spot_density": status.spot_density,
                "auto_class": status.auto_class,
            })

df_crystal = pd.DataFrame(crystal_results)

summary_crystal = df_crystal.groupby("distance_mm").agg(
    n_frames=("spot_density", "size"),
    mean_spot_density=("spot_density", "mean"),
    median_spot_density=("spot_density", "median"),
    n_classified_crystal=("auto_class", lambda s: (s == "crystal").sum()),
)
summary_crystal["pct_hit"] = 100 * summary_crystal["n_classified_crystal"] / summary_crystal["n_frames"]
print(summary_crystal)

---
## Step 3 — Combined summary: does distance help, hurt, or make no real difference?

Puts both halves side by side. The theory being tested is only supported if FEP SNR genuinely improves with distance AND crystal spot_density/hit-rate does not meaningfully degrade alongside it.

In [ ]:
print("="*70)
print("  FEP signal/noise vs distance")
print("="*70)
print(df_fep[["distance_mm", "ring_r_px", "signal", "noise", "snr"]].to_string(index=False))

print("\n" + "="*70)
print("  Crystal spot_density vs distance")
print("="*70)
print(summary_crystal)

print(
    "\nCAVEAT: this is a STATIC trial - informs which distance is most promising for real "
    "flowing beamtime, not a final proof of real-flow performance at each distance. Sample "
    "sizes at 100mm/200mm may be smaller than 50mm - treat any distance with low n as "
    "provisional."
)

In [ ]:
import matplotlib.pyplot as plt

# 1. Does the FEP ring visually shrink/weaken across distances, matching the numbers?
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, dist in zip(axes, [50, 100, 200]):
    numbers = collection_numbers_for("FEP_empty", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    frame = _load_pixium_frame(files[0])
    if frame.ndim == 3:
        frame = frame.mean(axis=0)
    vmin = max(1, np.percentile(frame[frame>0], 1))
    ax.imshow(frame, origin="lower", norm=LogNorm(vmin=vmin, vmax=frame.max()), cmap="viridis")
    ax.set_title(f"{dist}mm — FEP_empty")
plt.tight_layout()
plt.show()

# 2. Do confirmed crystal hits still look clean/resolved at 100mm and 200mm,
# same visual standard as the Swish reference from the technique comparison?
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, dist in zip(axes, [100, 200]):
    numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    hit_row = df_crystal[(df_crystal["distance_mm"] == dist) & (df_crystal["auto_class"] == "crystal")].iloc[0]
    fp = [f for f in files if hit_row["file"].replace("i11-1-","").replace(".nxs","") in f][0]

    frame = _load_pixium_frame(fp)   # load whole file, no frame_index kwarg
    if frame.ndim == 3:
        frame = frame[hit_row["sub_frame"]]   # index the sub-frame afterward

    vmin = max(1, np.percentile(frame[frame>0], 1))
    ax.imshow(frame, origin="lower", norm=LogNorm(vmin=vmin, vmax=frame.max()), cmap="viridis")
    ax.set_title(f"{dist}mm — confirmed crystal hit\n{hit_row['file']}, spot_density={hit_row['spot_density']:.5f}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, dist, hit_num in zip(axes, [100, 200], ["145413", "145479"]):
    numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    fp = [f for f in files if hit_num in f][0]
    hit_row = df_crystal[(df_crystal["distance_mm"] == dist) & (df_crystal["file"].str.contains(hit_num))].iloc[0]

    frame = _load_pixium_frame(fp)
    if frame.ndim == 3:
        frame = frame[hit_row["sub_frame"]]

    zoom = frame[800:1400, 800:1400]   # same crop window used for the 50mm Swish reference
    vmin = max(1, np.percentile(zoom[zoom>0], 1))
    ax.imshow(zoom, origin="lower", norm=LogNorm(vmin=vmin, vmax=zoom.max()), cmap="viridis")
    ax.set_title(f"{dist}mm zoomed — spot_density={hit_row['spot_density']:.5f}")
plt.tight_layout()
plt.show()

In [ ]:
def spot_diagnostics_v2(fp, dist, sub_frame, geom, crop=(800, 1400, 800, 1400), threshold_sigma=5):
    frame = _load_pixium_frame(fp)
    if frame.ndim == 3:
        frame = frame[sub_frame]

    mask = geom["mask"]
    arr = frame.astype(np.float64)
    arr[mask.astype(bool)] = 0.0

    rows, cols = arr.shape
    cy, cx = geom["cy"], geom["cx"]
    y_idx, x_idx = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x_idx - cx) ** 2 + (y_idx - cy) ** 2).astype(np.int32)
    r_max = int(r_map.max())

    radial_sum = np.bincount(r_map.ravel(), weights=arr.ravel(), minlength=r_max + 1)
    radial_cnt = np.bincount(r_map.ravel(), minlength=r_max + 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        radial_mean = np.where(radial_cnt > 0, radial_sum / radial_cnt, 0.0)

    residual = arr - radial_mean[r_map]   # ring/gradient removed properly
    r0, r1, c0, c1 = crop
    residual_zoom = residual[r0:r1, c0:c1]

    noise = _mad_sigma(residual_zoom.ravel())
    threshold = threshold_sigma * noise

    spot_mask = residual_zoom > threshold
    labeled, n_spots = ndimage.label(spot_mask)
    sizes = ndimage.sum(spot_mask, labeled, range(1, n_spots + 1))
    brightness = ndimage.maximum(residual_zoom, labeled, range(1, n_spots + 1))

    print(f"  n_distinct_spots = {n_spots}")
    if n_spots > 0:
        print(f"  mean spot size (px) = {sizes.mean():.1f}")
        print(f"  mean peak brightness above bg = {brightness.mean():.1f}")
    return n_spots, sizes, brightness

fp_100 = [f for f in files_for_numbers(collection_numbers_for("Gly_AS_tri", 100, exposure_code="Swish")) if "145413" in f][0]
fp_200 = [f for f in files_for_numbers(collection_numbers_for("Gly_AS_tri", 200, exposure_code="Swish")) if "145479" in f][0]

print("100mm:")
spot_diagnostics_v2(fp_100, 100, 0, geometries[100])
print("\n200mm:")
spot_diagnostics_v2(fp_200, 200, 0, geometries[200])

In [ ]:
def mean_contrast_for_distance(dist, geom, crop=(800, 1400, 800, 1400), threshold_sigma=5):
    hits = df_crystal[(df_crystal["distance_mm"] == dist) & (df_crystal["auto_class"] == "crystal")]
    all_brightness = []
    for _, hit_row in hits.iterrows():
        numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
        files = files_for_numbers(numbers)
        fp = [f for f in files if hit_row["file"].replace("i11-1-","").replace(".nxs","") in f][0]
        n_spots, sizes, brightness = spot_diagnostics_v2(fp, dist, hit_row["sub_frame"], geom, crop, threshold_sigma)
        if n_spots > 0:
            all_brightness.extend(brightness.tolist())
    return all_brightness

for dist in [50, 100, 200]:
    b = mean_contrast_for_distance(dist, geometries[dist])
    print(f"{dist}mm: n_hit_frames_checked, mean spot contrast = {np.mean(b):.1f}  (n_spots_total={len(b)})")

In [ ]:
def distance_from_poni(poni_path):
    """Read the raw Distance field (metres) from a .poni file."""
    with open(poni_path, "r") as f:
        for line in f:
            if line.strip().startswith("Distance:"):
                return float(line.split(":", 1)[1].strip())
    return None

# Add true distance (metres) into each geometry dict
for dist in [50, 100, 200]:
    geometries[dist]["distance_m"] = distance_from_poni(PONI_PATHS[dist])

def spot_diagnostics_annulus(fp, dist, sub_frame, geom, tth_lo_deg=7.0, tth_hi_deg=15.0, threshold_sigma=5):
    """Same radial-background-subtracted spot detection as before, but
    restricted to a fixed 2θ ANNULUS (not a fixed pixel box) - fair
    across distances since it covers the same real angular range and
    the same full 360°, not one quadrant."""
    frame = _load_pixium_frame(fp)
    if frame.ndim == 3:
        frame = frame[sub_frame]

    mask = geom["mask"]
    arr = frame.astype(np.float64)
    arr[mask.astype(bool)] = 0.0

    rows, cols = arr.shape
    cy, cx = geom["cy"], geom["cx"]
    y_idx, x_idx = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x_idx - cx) ** 2 + (y_idx - cy) ** 2).astype(np.int32)
    r_max = int(r_map.max())

    radial_sum = np.bincount(r_map.ravel(), weights=arr.ravel(), minlength=r_max + 1)
    radial_cnt = np.bincount(r_map.ravel(), minlength=r_max + 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        radial_mean = np.where(radial_cnt > 0, radial_sum / radial_cnt, 0.0)
    residual = arr - radial_mean[r_map]

    # Convert the fixed 2θ range to THIS geometry's pixel radius range
    pixel_size = 148e-6
    r_lo_px = int(geom["distance_m"] * np.tan(np.radians(tth_lo_deg)) / pixel_size)
    r_hi_px = int(geom["distance_m"] * np.tan(np.radians(tth_hi_deg)) / pixel_size)

    annulus_mask = (r_map >= r_lo_px) & (r_map <= r_hi_px) & ~mask.astype(bool)
    annulus_area = int(annulus_mask.sum())

    residual_valid = np.where(annulus_mask, residual, 0.0)
    noise = _mad_sigma(residual[annulus_mask])
    threshold = threshold_sigma * noise

    spot_mask = (residual_valid > threshold) & annulus_mask
    labeled, n_spots = ndimage.label(spot_mask)
    sizes = ndimage.sum(spot_mask, labeled, range(1, n_spots + 1)) if n_spots > 0 else np.array([])
    brightness = ndimage.maximum(residual, labeled, range(1, n_spots + 1)) if n_spots > 0 else np.array([])

    return n_spots, annulus_area, sizes, brightness

def mean_stats_for_distance_annulus(dist, geom):
    hits = df_crystal[(df_crystal["distance_mm"] == dist) & (df_crystal["auto_class"] == "crystal")]
    all_brightness, all_counts, all_areas = [], [], []
    for _, hit_row in hits.iterrows():
        numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
        files = files_for_numbers(numbers)
        fp = [f for f in files if hit_row["file"].replace("i11-1-","").replace(".nxs","") in f][0]
        n_spots, area, sizes, brightness = spot_diagnostics_annulus(fp, dist, hit_row["sub_frame"], geom)
        all_counts.append(n_spots)
        all_areas.append(area)
        if n_spots > 0:
            all_brightness.extend(brightness.tolist())
    density_per_1000px = 1000 * np.mean(all_counts) / np.mean(all_areas)
    return np.mean(all_counts), density_per_1000px, np.mean(all_brightness)

print(f"{'Distance':<10} {'Mean spot count':>16} {'Density/1000px':>16} {'Mean contrast':>15}")
print("-" * 60)
for dist in [50, 100, 200]:
    n, density, contrast = mean_stats_for_distance_annulus(dist, geometries[dist])
    print(f"{dist:<10} {n:>16.1f} {density:>16.3f} {contrast:>15.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Confirmed hit for each distance - reuse the 50mm reference from the
# Technique Comparison Report (i11-1-145302.nxs) for consistency across reports
hit_numbers = {50: "145302", 100: "145413", 200: "145479"}

for ax, dist in zip(axes, [50, 100, 200]):
    numbers = collection_numbers_for("Gly_AS_tri", dist, exposure_code="Swish")
    files = files_for_numbers(numbers)
    hit_row = df_crystal[
        (df_crystal["distance_mm"] == dist) &
        (df_crystal["file"].str.contains(hit_numbers[dist]))
    ].iloc[0]
    fp = [f for f in files if hit_numbers[dist] in f][0]

    frame = _load_pixium_frame(fp)
    if frame.ndim == 3:
        frame = frame[hit_row["sub_frame"]]

    zoom = frame[800:1400, 800:1400]   # same fixed crop as the existing 100/200mm figure
    vmin = max(1, np.percentile(zoom[zoom>0], 1))
    ax.imshow(zoom, origin="lower", norm=LogNorm(vmin=vmin, vmax=zoom.max()), cmap="viridis")
    ax.set_title(f"{dist}mm zoomed\n{hit_row['file']}, spot_density={hit_row['spot_density']:.5f}")

plt.tight_layout()
plt.show()